# P3｜PANNs Cnn14 Native Encoder

**状态：production adapter Code/合成 CPU contract READY；本地 source/checkpoint/dependency Asset HOLD；L40 CUDA 与实验审批 HOLD。** 目的：四数据集共用 frozen PANNs Cnn14_16k window encoder、candidate dimension adapter 与同一个 shared projector；native units/heads 不统一。

## 唯一变量、四数据集 lane、输入输出与科学 gate

matched comparator=P1；四数据集都使用 2.0 s / 1.0 s source-time windows。PANNs Cnn14 原生 embedding 是 2048-d，因此 package 明确包含一个跨四数据集共享的 trainable LayerNorm(2048)+bias-free Linear(2048→768)，其后仍是与 P1 相同的 shared Linear 768→256。该额外 adapter 使 P3−P1 只能作 package-level comparison。

ICBHI cycle、SPRSound event、KAUH recording 做 masked window aggregation；HF 保留 [B,K,256] temporal sequence 与 I/E/CAS/DAS native head。实现入口为 `baseline.multidataset_pipeline.panns_window_encoder` 与统一 `adapter_factory`；固定使用官方 16 kHz `Cnn14_16k_mAP=0.438.pth` recipe，不做未声明重采样。执行前仍需获得并冻结本地官方 source revision、checkpoint SHA、torchlibrosa dependency，完成真实 CPU/L40 smoke、approval 与 independent verifier；不得下载未授权资产、改 split或读取 outer/test。

In [ ]:
from baseline.multidataset_pipeline.preflight import P1_P5_SELECTION_RULE, P1_P5_UPDATE_BUDGET, freeze_receipt
from baseline.multidataset_pipeline.adapter_factory import AdapterFactoryConfig, build_production_adapter
from baseline.multidataset_pipeline.real_subtrain_provider import build_frozen_provider_index, build_real_subtrain_preflight_batches
from baseline.multidataset_pipeline.train_shared_window import TrainingRunnerConfig
PIPELINE = {"id": "P3", "comparator": "P1", "only_change": "four_dataset_shared_encoder_package_AST_to_PANNs_Cnn14_with_2048_to_768_adapter", "seed": 20260728, "window_policy": "source_time_2s_window_1s_stride", "shared_projector_lanes": ["ICBHI", "SPRSound", "HF", "KAUH"], "split_policy": "match_P1_frozen_provider", "provider_schema": "real_frozen_provider_identity_v2", "runner_schema": "shared_window_training_v5", "batch_size": 8, "update_budget": P1_P5_UPDATE_BUDGET, "validation_interval_updates": 1725, "selection": P1_P5_SELECTION_RULE, "runner": "baseline.multidataset_pipeline.train_shared_window", "output_dir": "result/reproduce/P3_shared_window_seed20260728", "receipt_path": "result/reproduce/P3_shared_window_seed20260728/<phase>/<approval_receipt_sha256>/run_receipt.json"}
PREFLIGHT = freeze_receipt()
raise RuntimeError("P3 asset HOLD until pinned local official source/checkpoint/dependency, real CPU smoke, L40 zero-update preflight, approval, and verifier pass")
DRY_RUN_PLAN = {"pipeline": PIPELINE, "execute": False}


## Receipt、结果表与 claim boundary

receipt：pipeline/config/manifest/checkpoint/window-adapter hashes、2048→768 parameter scope、P1 parity、four-dataset unit/window counts、HF target semantics、seed/updates、native metrics、selection、verifier warnings。

| Comparison | Result | Decision |
|---|---:|---|
| P3−P1，按 native task | Not run | 未判定 |

**Test Result = Not run。Decision = HOLD。只解释四数据集 PANNs+dimension-adapter package；不是统一标签任务。**